In [1]:
import pandas as pd

df = pd.read_csv("../data/DataSet.csv")

df.head()

,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset
0,Marketing Intern,"US, NY, New York",Marketing,NaN,"<h3>We're Food52, and we've created a groundbr...","<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,f,t,f,Other,Internship,NaN,NaN,Marketing,f,f
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"<h3>90 Seconds, the worlds Cloud Video Product...",<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,<h3><b>What you will get from us</b></h3>\r\n<...,f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,f,f
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,<h3></h3>\r\n<p>Valor Services provides Workfo...,"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,f,t,f,NaN,NaN,NaN,NaN,NaN,f,f
3,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,<p>Our passion for improving quality of life t...,<p><b>THE COMPANY: ESRI – Environmental System...,<ul>\r\n<li>\r\n<b>EDUCATION: </b>Bachelor’s o...,<p>Our culture is anything but corporate—we ha...,f,t,f,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,f,f
4,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,<p>SpotSource Solutions LLC is a Global Human ...,<p><b>JOB TITLE:</b> Itemization Review Manage...,<p><b>QUALIFICATIONS:</b></p>\r\n<ul>\r\n<li>R...,<p>Full Benefits Offered</p>,f,t,t,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,f,f


In [2]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 17880
Columns: 18


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   title                17880 non-null  str  
 1   location             17534 non-null  str  
 2   department           6333 non-null   str  
 3   salary_range         2868 non-null   str  
 4   company_profile      14572 non-null  str  
 5   description          17880 non-null  str  
 6   requirements         15191 non-null  str  
 7   benefits             10684 non-null  str  
 8   telecommuting        17880 non-null  str  
 9   has_company_logo     17880 non-null  str  
 10  has_questions        17880 non-null  str  
 11  employment_type      14409 non-null  str  
 12  required_experience  10830 non-null  str  
 13  required_education   9775 non-null   str  
 14  industry             12977 non-null  str  
 15  function             11425 non-null  str  
 16  fraudulent           17880 non-nu

In [4]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

missing

salary_range           15012
department             11547
required_education      8105
benefits                7196
required_experience     7050
function                6455
industry                4903
employment_type         3471
company_profile         3308
requirements            2689
location                 346
dtype: int64

In [5]:
missing_percentage = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_percentage

salary_range           83.959732
department             64.580537
required_education     45.329978
benefits               40.246085
required_experience    39.429530
function               36.101790
industry               27.421700
employment_type        19.412752
company_profile        18.501119
requirements           15.039150
location                1.935123
title                   0.000000
description             0.000000
telecommuting           0.000000
has_company_logo        0.000000
has_questions           0.000000
fraudulent              0.000000
in_balanced_dataset     0.000000
dtype: float64

In [6]:
df_clean = df.copy()

# Remove exact duplicate job postings
df_clean = df_clean.drop_duplicates()

# Remove dataset-construction column
df_clean = df_clean.drop(columns=["in_balanced_dataset"])

print("Original rows:", len(df))
print("Clean rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

Original rows: 17880
Clean rows: 17645
Columns: 17


In [7]:
from bs4 import BeautifulSoup

def clean_html(text):
    if pd.isna(text):
        return ""
    
    soup = BeautifulSoup(str(text), "html.parser")
    return soup.get_text(" ", strip=True)

In [8]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

for column in text_columns:
    df_clean[column] = df_clean[column].apply(clean_html)

In [9]:
print(df_clean["description"].iloc[0])

Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and curated recipe hub, is currently interviewing full- and part-time unpaid interns to work in a small team of editors, executives, and developers in its New York City headquarters. Reproducing and/or repackaging existing Food52 content for a number of partner sites, such as Huffington Post, Yahoo, Buzzfeed, and more in their various content management systems Researching blogs and websites for the Provisions by Food52 Affiliate Program Assisting in day-to-day affiliate program support, such as screening affiliates and assisting in any affiliate inquiries Supporting with PR & Events when needed Helping with office administrative work, such as filing, mailing, and preparing for meetings Working with developers to document bugs and suggest improvements to the site Supporting the marketing and executive staff


In [24]:
def clean_text(text):
    text = str(text).lower()

    # Remove long hexadecimal/hash-like tokens
    text = re.sub(r'\b[a-f0-9]{20,}\b', ' ', text)

    # Remove isolated short number tokens
    text = re.sub(r'\b\d{1,2}\b', ' ', text)

    # Remove very long digit-only tokens
    text = re.sub(r'\b\d{6,}\b', ' ', text)

    # Replace punctuation with spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [25]:
for column in text_columns:
    df_clean[column] = df_clean[column].apply(clean_text)

In [26]:
df_clean["combined_text"] = (
    df_clean[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

In [27]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

df_clean["combined_text"] = (
    df_clean[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

In [28]:
print(df_clean["combined_text"].iloc[0])

marketing intern we re food52 and we ve created a groundbreaking and award winning cooking site we support connect and celebrate home cooks and give them everything they need in one place we have a top editorial business and engineering team we re focused on using technology to find new and better ways to connect people around their specific food interests and to offer them superb highly curated information about food and cooking we attract the most talented home cooks and contributors in the country we also publish well known professionals like mario batali gwyneth paltrow and danny meyer and we have partnerships with whole foods market and random house food52 has been named the best food website by the james beard foundation and iacp and has been featured in the new york times npr pando daily techcrunch and on the today show we re located in chelsea in new york city food52 a fast growing james beard award winning online food community and crowd sourced and curated recipe hub is curre

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
)

X_text = tfidf.fit_transform(df_clean["combined_text"])

print("Number of job postings:", X_text.shape[0])
print("Number of TF-IDF features:", X_text.shape[1])
print(tfidf.get_feature_names_out()[:50])

Number of job postings: 17645
Number of TF-IDF features: 50000
['aa' 'aaa' 'aac' 'aan' 'aan de' 'aan die' 'ab' 'abap' 'abc' 'abc sales'
 'abc supply' 'aberdeen' 'abilities' 'abilities ability' 'abilities and'
 'abilities are' 'abilities experience' 'abilities strong' 'abilities to'
 'ability' 'ability ability' 'ability and' 'ability bend' 'ability in'
 'ability of' 'ability required' 'ability to' 'able' 'able and' 'able to'
 'aboard' 'about' 'about all' 'about and' 'about any' 'about apex'
 'about argenta' 'about at' 'about being' 'about both' 'about bringing'
 'about building' 'about changing' 'about countries' 'about creating'
 'about customer' 'about delivering' 'about design' 'about developing'
 'about digital']


In [32]:
print(df_clean["fraudulent"].value_counts())

fraudulent
f    16787
t      858
Name: count, dtype: int64


In [33]:
print(df_clean["fraudulent"].value_counts(normalize=True) * 100)

fraudulent
f    95.137433
t     4.862567
Name: proportion, dtype: float64


In [34]:
X = df_clean["combined_text"]
y = df_clean["fraudulent"].map({"f": 0, "t": 1})

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (17645,)
y shape: (17645,)


In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Fraud in training:", y_train.sum())
print("Fraud in testing:", y_test.sum())

Training samples: 14116
Testing samples: 3529
Fraud in training: 686
Fraud in testing: 172


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (14116, 50000)
Testing TF-IDF shape: (3529, 50000)


In [38]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Model training completed!")

Model training completed!


In [39]:
y_pred = model.predict(X_test_tfidf)
y_prob = model.predict_proba(X_test_tfidf)[:, 1]

print("Predictions completed!")

Predictions completed!


In [40]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Legitimate", "Fraudulent"]
))

              precision    recall  f1-score   support

  Legitimate       0.99      0.98      0.99      3357
  Fraudulent       0.69      0.84      0.76       172

    accuracy                           0.97      3529
   macro avg       0.84      0.91      0.87      3529
weighted avg       0.98      0.97      0.97      3529



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[3291   66]
 [  27  145]]


In [43]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, y_prob)

print("PR-AUC:", round(pr_auc, 3))

PR-AUC: 0.882


In [44]:
import pandas as pd

feature_names = tfidf.get_feature_names_out()
coefficients = model.coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

fraud_features = feature_importance.sort_values(
    "coefficient",
    ascending=False
)

print("Top features associated with fraud:")
print(fraud_features.head(20))

Top features associated with fraud:
                         feature  coefficient
25141               listed above     3.728912
11783                   database     3.286906
6484             benchmark shows     3.147487
46482                  using and     3.139654
40808                 succeed we     3.102970
181                   accidental     3.069849
5238                  assistants     3.049921
652    administration experience     2.979108
4359            appointments etc     2.892662
14724           engineering team     2.883095
25137              list includes     2.609152
15077    environment collections     2.608363
27393             monitoring the     2.603498
35702                  record of     2.515337
244               accountable to     2.504784
8996                    clicking     2.492179
32694              photographers     2.466020
13954                  eatads is     2.460730
29783                       ohio     2.422799
21975             increase sales     2.30934

In [45]:
print("\nTop features associated with legitimate jobs:")
print(fraud_features.tail(20).sort_values("coefficient"))


Top features associated with legitimate jobs:
                   feature  coefficient
31116             our blog    -4.046104
28734    of administrative    -3.115130
21121           in america    -2.375866
26312   marketing director    -2.272231
13075           diligently    -2.212028
23541        it department    -2.125917
47538  wearables including    -2.025724
9746   companies including    -1.991063
14787      english written    -1.911508
19316       growing online    -1.770420
1497             amount of    -1.755353
23147               is and    -1.708931
41590              team an    -1.679045
37888            secret is    -1.648787
31                   about    -1.642857
26684          media sales    -1.631037
9090        clients assist    -1.622663
5940        based business    -1.617089
35832     reduce workplace    -1.542636
11218        creative ways    -1.541878


In [46]:
df_clean["company_profile_missing"] = (
    df_clean["company_profile"].fillna("").str.strip() == ""
).astype(int)

df_clean["requirements_missing"] = (
    df_clean["requirements"].fillna("").str.strip() == ""
).astype(int)

df_clean["benefits_missing"] = (
    df_clean["benefits"].fillna("").str.strip() == ""
).astype(int)

print(df_clean[
    ["company_profile_missing", "requirements_missing", "benefits_missing"]
].head())

   company_profile_missing  requirements_missing  benefits_missing
0                        0                     0                 1
1                        0                     0                 0
2                        0                     0                 1
3                        0                     0                 0
4                        0                     0                 0


In [47]:
binary_columns = [
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

for column in binary_columns:
    print(column)
    print(df_clean[column].value_counts(dropna=False))
    print()

telecommuting
telecommuting
f    16889
t      756
Name: count, dtype: int64

has_company_logo
has_company_logo
t    14011
f     3634
Name: count, dtype: int64

has_questions
has_questions
f    8969
t    8676
Name: count, dtype: int64



In [48]:
for column in binary_columns:
    df_clean[column] = df_clean[column].map({"f": 0, "t": 1})

print(df_clean[binary_columns].head())



   telecommuting  has_company_logo  has_questions
0              0                 1              0
1              0                 1              0
2              0                 1              0
3              0                 1              0
4              0                 1              1


In [49]:
categorical_columns = [
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
    "location"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print("Unique values:", df_clean[column].nunique())
    print("Missing:", df_clean[column].isna().sum())
    print(df_clean[column].value_counts(dropna=False).head(10))


--- employment_type ---
Unique values: 5
Missing: 3435
employment_type
Full-time    11457
NaN           3435
Contract      1517
Part-time      774
Temporary      237
Other          225
Name: count, dtype: int64

--- required_experience ---
Unique values: 7
Missing: 6976
required_experience
NaN                 6976
Mid-Senior level    3774
Entry level         2645
Associate           2274
Not Applicable      1077
Director             385
Internship           374
Executive            140
Name: count, dtype: int64

--- required_education ---
Unique values: 13
Missing: 8028
required_education
NaN                                  8028
Bachelor's Degree                    5107
High School or equivalent            2002
Unspecified                          1375
Master's Degree                       416
Associate Degree                      264
Certification                         165
Some College Coursework Completed     100
Professional                           73
Vocational               

In [50]:
categorical_data = df_clean[categorical_columns].copy()

categorical_data = categorical_data.fillna("MISSING")

print(categorical_data.head())

  employment_type required_experience required_education  \
0           Other          Internship            MISSING   
1       Full-time      Not Applicable            MISSING   
2         MISSING             MISSING            MISSING   
3       Full-time    Mid-Senior level  Bachelor's Degree   
4       Full-time    Mid-Senior level  Bachelor's Degree   

                    industry              function            location  
0                    MISSING             Marketing    US, NY, New York  
1  Marketing and Advertising      Customer Service      NZ, , Auckland  
2                    MISSING               MISSING       US, IA, Wever  
3          Computer Software                 Sales  US, DC, Washington  
4     Hospital & Health Care  Health Care Provider  US, FL, Fort Worth  


In [51]:
print(categorical_data.isnull().sum())

employment_type        0
required_experience    0
required_education     0
industry               0
function               0
location               0
dtype: int64


In [52]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_categorical = encoder.fit_transform(categorical_data)

print("Categorical feature matrix shape:", X_categorical.shape)

Categorical feature matrix shape: (17645, 3304)


In [53]:
X_cat_train = categorical_data.loc[X_train.index]
X_cat_test = categorical_data.loc[X_test.index]

print("Categorical training shape:", X_cat_train.shape)
print("Categorical testing shape:", X_cat_test.shape)

Categorical training shape: (14116, 6)
Categorical testing shape: (3529, 6)


In [54]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_cat_train_encoded = encoder.fit_transform(X_cat_train)
X_cat_test_encoded = encoder.transform(X_cat_test)

print("Encoded training shape:", X_cat_train_encoded.shape)
print("Encoded testing shape:", X_cat_test_encoded.shape)

Encoded training shape: (14116, 2957)
Encoded testing shape: (3529, 2957)


In [55]:
from scipy.sparse import hstack

X_train_hybrid = hstack([
    X_train_tfidf,
    X_cat_train_encoded
])

X_test_hybrid = hstack([
    X_test_tfidf,
    X_cat_test_encoded
])

print("Hybrid training shape:", X_train_hybrid.shape)
print("Hybrid testing shape:", X_test_hybrid.shape)

Hybrid training shape: (14116, 52957)
Hybrid testing shape: (3529, 52957)


In [56]:
from sklearn.linear_model import LogisticRegression

hybrid_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

hybrid_model.fit(X_train_hybrid, y_train)

print("Hybrid model training completed!")

Hybrid model training completed!


In [57]:
y_hybrid_pred = hybrid_model.predict(X_test_hybrid)
y_hybrid_prob = hybrid_model.predict_proba(X_test_hybrid)[:, 1]

print("Hybrid predictions completed!")

Hybrid predictions completed!


In [58]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_hybrid_pred,
    target_names=["Legitimate", "Fraudulent"]
))

              precision    recall  f1-score   support

  Legitimate       0.99      0.98      0.99      3357
  Fraudulent       0.68      0.87      0.76       172

    accuracy                           0.97      3529
   macro avg       0.84      0.93      0.87      3529
weighted avg       0.98      0.97      0.98      3529



In [59]:
from sklearn.metrics import average_precision_score

hybrid_pr_auc = average_precision_score(y_test, y_hybrid_prob)

print("Hybrid PR-AUC:", round(hybrid_pr_auc, 3))

Hybrid PR-AUC: 0.845


In [60]:
from sklearn.metrics import confusion_matrix

hybrid_cm = confusion_matrix(y_test, y_hybrid_pred)

print(hybrid_cm)

[[3286   71]
 [  22  150]]


In [61]:
baseline_results = {
    "Text-only Logistic Regression": {
        "Accuracy": 0.97,
        "Fraud Precision": 0.69,
        "Fraud Recall": 0.84,
        "Fraud F1": 0.76,
        "PR-AUC": 0.882
    },
    "Hybrid Logistic Regression": {
        "Accuracy": 0.97,
        "Fraud Precision": 0.68,
        "Fraud Recall": 0.87,
        "Fraud F1": 0.76,
        "PR-AUC": 0.845
    }
}

baseline_results

{'Text-only Logistic Regression': {'Accuracy': 0.97,
  'Fraud Precision': 0.69,
  'Fraud Recall': 0.84,
  'Fraud F1': 0.76,
  'PR-AUC': 0.882},
 'Hybrid Logistic Regression': {'Accuracy': 0.97,
  'Fraud Precision': 0.68,
  'Fraud Recall': 0.87,
  'Fraud F1': 0.76,
  'PR-AUC': 0.845}}

In [62]:
def create_weighted_text(row):
    title = row["title"]
    company = row["company_profile"]
    description = row["description"]
    requirements = row["requirements"]
    benefits = row["benefits"]

    return (
        f"{title} {title} "
        f"{company} "
        f"{description} "
        f"{requirements} {requirements} "
        f"{benefits}"
    )

df_clean["weighted_text"] = df_clean.apply(create_weighted_text, axis=1)

print(df_clean["weighted_text"].iloc[0])

marketing intern marketing intern we re food52 and we ve created a groundbreaking and award winning cooking site we support connect and celebrate home cooks and give them everything they need in one place we have a top editorial business and engineering team we re focused on using technology to find new and better ways to connect people around their specific food interests and to offer them superb highly curated information about food and cooking we attract the most talented home cooks and contributors in the country we also publish well known professionals like mario batali gwyneth paltrow and danny meyer and we have partnerships with whole foods market and random house food52 has been named the best food website by the james beard foundation and iacp and has been featured in the new york times npr pando daily techcrunch and on the today show we re located in chelsea in new york city food52 a fast growing james beard award winning online food community and crowd sourced and curated re

In [63]:
X_weighted = df_clean["weighted_text"]

X_weighted_train = X_weighted.loc[X_train.index]
X_weighted_test = X_weighted.loc[X_test.index]

print("Weighted training samples:", len(X_weighted_train))
print("Weighted testing samples:", len(X_weighted_test))

Weighted training samples: 14116
Weighted testing samples: 3529


In [64]:
weighted_tfidf = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
)

X_weighted_train_tfidf = weighted_tfidf.fit_transform(X_weighted_train)
X_weighted_test_tfidf = weighted_tfidf.transform(X_weighted_test)

print("Weighted training TF-IDF shape:", X_weighted_train_tfidf.shape)
print("Weighted testing TF-IDF shape:", X_weighted_test_tfidf.shape)

Weighted training TF-IDF shape: (14116, 50000)
Weighted testing TF-IDF shape: (3529, 50000)


In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# 1. Get weighted text
X_weighted = df_clean["weighted_text"]

# 2. Use the SAME train/test split indices
X_weighted_train = X_weighted.loc[X_train.index]
X_weighted_test = X_weighted.loc[X_test.index]

# 3. Fit TF-IDF ONLY on training data
weighted_tfidf = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
)

X_weighted_train_tfidf = weighted_tfidf.fit_transform(X_weighted_train)
X_weighted_test_tfidf = weighted_tfidf.transform(X_weighted_test)

# 4. Train model
weighted_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

weighted_model.fit(X_weighted_train_tfidf, y_train)

# 5. Predictions
y_weighted_pred = weighted_model.predict(X_weighted_test_tfidf)
y_weighted_prob = weighted_model.predict_proba(
    X_weighted_test_tfidf
)[:, 1]

# 6. Evaluation
print("=== Weighted Text Model ===\n")

print(classification_report(
    y_test,
    y_weighted_pred,
    target_names=["Legitimate", "Fraudulent"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_weighted_pred))

print("\nPR-AUC:", round(
    average_precision_score(y_test, y_weighted_prob),
    3
))

=== Weighted Text Model ===

              precision    recall  f1-score   support

  Legitimate       0.99      0.98      0.99      3357
  Fraudulent       0.69      0.84      0.76       172

    accuracy                           0.97      3529
   macro avg       0.84      0.91      0.87      3529
weighted avg       0.98      0.97      0.98      3529

Confusion Matrix:
[[3293   64]
 [  28  144]]

PR-AUC: 0.873


In [66]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\Jobshield\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [67]:
# Generate semantic embeddings for training and testing data

X_train_embeddings = embedding_model.encode(
    X_train.tolist(),
    show_progress_bar=True,
    batch_size=32
)

X_test_embeddings = embedding_model.encode(
    X_test.tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Training embeddings shape:", X_train_embeddings.shape)
print("Testing embeddings shape:", X_test_embeddings.shape)

Batches:   0%|          | 0/442 [00:00<?, ?it/s]

Batches:   0%|          | 0/111 [00:00<?, ?it/s]

Training embeddings shape: (14116, 384)
Testing embeddings shape: (3529, 384)


In [68]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Train classifier on semantic embeddings
embedding_model_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

embedding_model_classifier.fit(
    X_train_embeddings,
    y_train
)

# Predictions
y_embedding_pred = embedding_model_classifier.predict(
    X_test_embeddings
)

y_embedding_prob = embedding_model_classifier.predict_proba(
    X_test_embeddings
)[:, 1]

# Evaluation
print("=== Semantic Embedding Model ===\n")

print(classification_report(
    y_test,
    y_embedding_pred,
    target_names=["Legitimate", "Fraudulent"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_embedding_pred))

print("\nPR-AUC:",
      round(average_precision_score(y_test, y_embedding_prob), 3))

=== Semantic Embedding Model ===

              precision    recall  f1-score   support

  Legitimate       0.99      0.88      0.93      3357
  Fraudulent       0.27      0.85      0.41       172

    accuracy                           0.88      3529
   macro avg       0.63      0.87      0.67      3529
weighted avg       0.96      0.88      0.91      3529

Confusion Matrix:
[[2960  397]
 [  26  146]]

PR-AUC: 0.536


In [69]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for threshold in thresholds:
    y_threshold_pred = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_threshold_pred, zero_division=0)
    recall = recall_score(y_test, y_threshold_pred, zero_division=0)
    f1 = f1_score(y_test, y_threshold_pred, zero_division=0)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Fraud Precision": round(precision, 3),
        "Fraud Recall": round(recall, 3),
        "Fraud F1": round(f1, 3)
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

 Threshold  Fraud Precision  Fraud Recall  Fraud F1
      0.10            0.144         1.000     0.251
      0.15            0.209         0.977     0.344
      0.20            0.278         0.971     0.432
      0.25            0.351         0.965     0.515
      0.30            0.437         0.942     0.597
      0.35            0.507         0.901     0.649
      0.40            0.593         0.890     0.712
      0.45            0.645         0.878     0.744
      0.50            0.687         0.843     0.757
      0.55            0.718         0.814     0.763
      0.60            0.789         0.802     0.795
      0.65            0.811         0.797     0.804
      0.70            0.848         0.779     0.812
      0.75            0.878         0.750     0.809
      0.80            0.925         0.721     0.810
      0.85            0.941         0.645     0.766
      0.90            0.958         0.529     0.682


In [70]:
from sklearn.model_selection import train_test_split

# Split the existing training data into training + validation
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=42
)

print("Final training samples:", len(X_train_final))
print("Validation samples:", len(X_val))
print("Final training fraud:", y_train_final.sum())
print("Validation fraud:", y_val.sum())
print("Test samples (untouched):", len(X_test))
print("Test fraud (untouched):", y_test.sum())

Final training samples: 11292
Validation samples: 2824
Final training fraud: 549
Validation fraud: 137
Test samples (untouched): 3529
Test fraud (untouched): 172


In [71]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score
import numpy as np
import pandas as pd

# -----------------------------
# 1. TF-IDF
# -----------------------------
final_tfidf = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
)

X_train_final_tfidf = final_tfidf.fit_transform(X_train_final)
X_val_tfidf = final_tfidf.transform(X_val)

print("Training TF-IDF shape:", X_train_final_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)


# -----------------------------
# 2. Train Logistic Regression
# -----------------------------
validation_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

validation_model.fit(X_train_final_tfidf, y_train_final)


# -----------------------------
# 3. Get validation probabilities
# -----------------------------
y_val_prob = validation_model.predict_proba(X_val_tfidf)[:, 1]

print("\nValidation PR-AUC:",
      round(average_precision_score(y_val, y_val_prob), 3))


# -----------------------------
# 4. Test different thresholds
# -----------------------------
thresholds = np.arange(0.10, 0.91, 0.05)

results = []

for threshold in thresholds:

    y_val_pred = (y_val_prob >= threshold).astype(int)

    report = classification_report(
        y_val,
        y_val_pred,
        output_dict=True,
        zero_division=0
    )

    results.append({
        "threshold": round(threshold, 2),
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1": report["1"]["f1-score"]
    })

threshold_results = pd.DataFrame(results)

print("\nValidation threshold results:")
display(threshold_results.round(3))


# -----------------------------
# 5. Select threshold by F1
# -----------------------------
best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = best_row["threshold"]

print("\nBest validation threshold:",
      best_threshold)

print("Precision:",
      round(best_row["precision"], 3))

print("Recall:",
      round(best_row["recall"], 3))

print("F1:",
      round(best_row["f1"], 3))

Training TF-IDF shape: (11292, 50000)
Validation TF-IDF shape: (2824, 50000)

Validation PR-AUC: 0.933

Validation threshold results:


,threshold,precision,recall,f1
0,0.10,0.130,0.993,0.230
1,0.15,0.212,0.993,0.349
2,0.20,0.306,0.985,0.467
3,0.25,0.401,0.978,0.569
4,0.30,0.468,0.971,0.632
5,0.35,0.542,0.949,0.690
6,0.40,0.615,0.934,0.742
7,0.45,0.702,0.927,0.799
8,0.50,0.776,0.912,0.839
9,0.55,0.819,0.891,0.853



Best validation threshold: 0.6
Precision: 0.895
Recall: 0.869
F1: 0.881


In [73]:
# ============================================
# FINAL MODEL TRAINING + UNTOUCHED TEST EVALUATION
# ============================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

# ------------------------------------------------
# 1. Freeze the threshold selected on validation
# ------------------------------------------------

final_threshold = 0.60

print("Frozen decision threshold:", final_threshold)


# ------------------------------------------------
# 2. Combine original training + validation data
# ------------------------------------------------

X_combined = pd.concat([X_train_final, X_val])
y_combined = pd.concat([y_train_final, y_val])

print("\nCombined training samples:", len(X_combined))
print("Combined training fraud:", y_combined.sum())


# ------------------------------------------------
# 3. Fit TF-IDF ONLY on combined training data
# ------------------------------------------------

final_tfidf = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
)

X_combined_tfidf = final_tfidf.fit_transform(X_combined)
X_test_final_tfidf = final_tfidf.transform(X_test)

print("\nFinal training TF-IDF shape:",
      X_combined_tfidf.shape)

print("Final test TF-IDF shape:",
      X_test_final_tfidf.shape)


# ------------------------------------------------
# 4. Train final Logistic Regression
# ------------------------------------------------

final_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

final_model.fit(X_combined_tfidf, y_combined)


# ------------------------------------------------
# 5. Generate test probabilities
# ------------------------------------------------

y_test_prob = final_model.predict_proba(
    X_test_final_tfidf
)[:, 1]


# ------------------------------------------------
# 6. Apply the FROZEN 0.60 threshold
# ------------------------------------------------

y_test_pred = (
    y_test_prob >= final_threshold
).astype(int)


# ------------------------------------------------
# 7. Final classification report
# ------------------------------------------------

print("\n========== FINAL TEST RESULTS ==========\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["Legitimate", "Fraud"],
        digits=3,
        zero_division=0
    )
)


# ------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------

cm = confusion_matrix(y_test, y_test_pred)

print("Confusion Matrix:")
print(cm)


# ------------------------------------------------
# 9. Ranking metrics
# ------------------------------------------------

pr_auc = average_precision_score(
    y_test,
    y_test_prob
)

roc_auc = roc_auc_score(
    y_test,
    y_test_prob
)

print("\nPR-AUC:", round(pr_auc, 3))
print("ROC-AUC:", round(roc_auc, 3))

Frozen decision threshold: 0.6

Combined training samples: 14116
Combined training fraud: 686

Final training TF-IDF shape: (14116, 50000)
Final test TF-IDF shape: (3529, 50000)

========== FINAL TEST RESULTS ==========

              precision    recall  f1-score   support

  Legitimate      0.990     0.989     0.989      3357
       Fraud      0.789     0.802     0.795       172

    accuracy                          0.980      3529
   macro avg      0.889     0.896     0.892      3529
weighted avg      0.980     0.980     0.980      3529

Confusion Matrix:
[[3320   37]
 [  34  138]]

PR-AUC: 0.882
ROC-AUC: 0.986


In [74]:
# ============================================
# JOBSHIELD MODEL EXPLAINABILITY
# Global feature importance from coefficients
# ============================================

import numpy as np
import pandas as pd

# Get feature names from the final TF-IDF vectorizer
feature_names = np.array(final_tfidf.get_feature_names_out())

# Get Logistic Regression coefficients
coefficients = final_model.coef_[0]

# --------------------------------------------
# Top features associated with FRAUD
# --------------------------------------------

top_fraud_indices = np.argsort(coefficients)[-20:][::-1]

fraud_features = pd.DataFrame({
    "feature": feature_names[top_fraud_indices],
    "coefficient": coefficients[top_fraud_indices]
})

print("Top 20 features associated with FRAUD:\n")
display(fraud_features.round(3))


# --------------------------------------------
# Top features associated with LEGITIMATE
# --------------------------------------------

top_legit_indices = np.argsort(coefficients)[:20]

legit_features = pd.DataFrame({
    "feature": feature_names[top_legit_indices],
    "coefficient": coefficients[top_legit_indices]
})

print("\nTop 20 features associated with LEGITIMATE:\n")
display(legit_features.round(3))

Top 20 features associated with FRAUD:



,feature,coefficient
0,link url,3.729
1,data entry,3.287
2,below link,3.147
3,using below,3.140
4,subsea,3.103
5,accion,3.070
6,assistant,3.050
7,administrative,2.979
8,apply using,2.893
9,engineering,2.883



Top 20 features associated with LEGITIMATE:



,feature,coefficient
0,our,-4.046
1,of,-3.115
2,in,-2.376
3,marketing,-2.272
4,digital,-2.212
5,it,-2.126
6,web,-2.026
7,companies,-1.991
8,english,-1.912
9,growing,-1.770


In [76]:
# ============================================
# JOBSHIELD — PER-JOB EXPLAINABILITY
# ============================================

import numpy as np
import pandas as pd

# Pick one job from the test set
sample_index = X_test.index[0]

sample_text = X_test.loc[sample_index]
sample_vector = X_test_final_tfidf[
    list(X_test.index).index(sample_index)
]

# Get feature names and model coefficients
feature_names = np.array(final_tfidf.get_feature_names_out())
coefficients = final_model.coef_[0]

# Convert sparse vector to array
feature_values = sample_vector.toarray().flatten()

# Calculate contribution of every feature
contributions = feature_values * coefficients

# Create explanation table
explanation = pd.DataFrame({
    "feature": feature_names,
    "tfidf_value": feature_values,
    "coefficient": coefficients,
    "contribution": contributions
})

# Keep only features actually present in this job
explanation = explanation[
    explanation["tfidf_value"] > 0
].copy()

# --------------------------------------------
# Features pushing toward FRAUD
# --------------------------------------------

fraud_signals = explanation[
    explanation["contribution"] > 0
].sort_values(
    "contribution",
    ascending=False
).head(15)

# --------------------------------------------
# Features pushing toward LEGITIMATE
# --------------------------------------------

legit_signals = explanation[
    explanation["contribution"] < 0
].sort_values(
    "contribution",
    ascending=True
).head(15)

# --------------------------------------------
# Prediction
# --------------------------------------------

sample_probability = final_model.predict_proba(
    sample_vector
)[0, 1]

sample_prediction = (
    "Potentially Fraudulent"
    if sample_probability >= final_threshold
    else "Likely Legitimate"
)

print("============================================")
print("JOBSHIELD SAMPLE PREDICTION")
print("============================================")

print("\nPrediction:", sample_prediction)
print("Fraud probability:", round(sample_probability, 3))
print("Decision threshold:", final_threshold)

print("\n--------------------------------------------")
print("TOP FRAUD-PUSHING FEATURES")
print("--------------------------------------------")

display(
    fraud_signals[
        ["feature", "tfidf_value", "coefficient", "contribution"]
    ].round(3)
)

print("\n--------------------------------------------")
print("TOP LEGITIMATE-PUSHING FEATURES")
print("--------------------------------------------")

display(
    legit_signals[
        ["feature", "tfidf_value", "coefficient", "contribution"]
    ].round(3)
)

JOBSHIELD SAMPLE PREDICTION

Prediction: Likely Legitimate
Fraud probability: 0.095
Decision threshold: 0.6

--------------------------------------------
TOP FRAUD-PUSHING FEATURES
--------------------------------------------


,feature,tfidf_value,coefficient,contribution
9797,company,0.088,1.361,0.119
13294,discounts,0.073,1.235,0.090
28395,no,0.027,2.302,0.062
22572,insurance,0.066,0.867,0.057
25137,link,0.022,2.609,0.057
7958,calling,0.064,0.819,0.053
7932,call,0.033,1.540,0.050
31852,paid,0.096,0.510,0.049
38315,service,0.036,1.319,0.048
44168,time,0.038,1.251,0.047



--------------------------------------------
TOP LEGITIMATE-PUSHING FEATURES
--------------------------------------------


,feature,tfidf_value,coefficient,contribution
31116,our,0.096,-4.046,-0.388
28734,of,0.066,-3.115,-0.206
1794,and,0.154,-1.147,-0.177
26312,marketing,0.057,-2.272,-0.129
21121,in,0.042,-2.376,-0.100
47320,we,0.064,-1.518,-0.098
28175,new,0.064,-1.331,-0.085
42477,the,0.074,-1.081,-0.080
17275,for,0.082,-0.859,-0.070
44310,to,0.048,-1.146,-0.055


In [77]:
# ============================================
# JOBSHIELD — HUMAN-READABLE MODEL SIGNALS
# ============================================

# Common words that are technically model features
# but are not useful to show users.
stop_words_for_explanation = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for",
    "from", "in", "is", "it", "of", "on", "or", "that",
    "the", "this", "to", "was", "we", "were", "with",
    "our", "your", "you", "their", "they", "will", "can",
    "has", "have", "had", "do", "does", "did", "not", "no"
}


def get_job_explanation(sample_index, top_n=10):
    """
    Generate human-readable model signals for one job.
    """

    # Locate the job in the test set
    test_position = list(X_test.index).index(sample_index)

    sample_vector = X_test_final_tfidf[test_position]

    # Model probability
    probability = final_model.predict_proba(sample_vector)[0, 1]

    # Prediction using frozen threshold
    prediction = (
        "Potentially Fraudulent"
        if probability >= final_threshold
        else "Likely Legitimate"
    )

    # Feature information
    feature_values = sample_vector.toarray().flatten()

    contributions = feature_values * coefficients

    explanation = pd.DataFrame({
        "feature": feature_names,
        "tfidf_value": feature_values,
        "coefficient": coefficients,
        "contribution": contributions
    })

    # Only features actually present in the job
    explanation = explanation[
        explanation["tfidf_value"] > 0
    ].copy()

    # Remove common/unhelpful words
    explanation["feature_words"] = (
        explanation["feature"]
        .str.lower()
        .str.split()
    )

    explanation = explanation[
        ~explanation["feature_words"].apply(
            lambda words: all(
                word in stop_words_for_explanation
                for word in words
            )
        )
    ].copy()

    # ----------------------------------------
    # Fraud-pushing signals
    # ----------------------------------------

    fraud_signals = explanation[
        explanation["contribution"] > 0
    ].sort_values(
        "contribution",
        ascending=False
    ).head(top_n)

    # ----------------------------------------
    # Legitimate-pushing signals
    # ----------------------------------------

    legit_signals = explanation[
        explanation["contribution"] < 0
    ].sort_values(
        "contribution"
    ).head(top_n)

    return {
        "probability": probability,
        "prediction": prediction,
        "fraud_signals": fraud_signals[
            ["feature", "contribution"]
        ].reset_index(drop=True),
        "legit_signals": legit_signals[
            ["feature", "contribution"]
        ].reset_index(drop=True)
    }


# ============================================
# TEST IT ON THE SAME JOB
# ============================================

sample_explanation = get_job_explanation(sample_index)

print("============================================")
print("JOBSHIELD EXPLANATION")
print("============================================")

print(
    "\nPrediction:",
    sample_explanation["prediction"]
)

print(
    "Fraud probability:",
    round(sample_explanation["probability"], 3)
)

print("\n🔴 Fraud-pushing signals:")
display(
    sample_explanation["fraud_signals"].round(3)
)

print("\n🟢 Legitimate-pushing signals:")
display(
    sample_explanation["legit_signals"].round(3)
)

JOBSHIELD EXPLANATION

Prediction: Likely Legitimate
Fraud probability: 0.095

🔴 Fraud-pushing signals:


,feature,contribution
0,company,0.119
1,discounts,0.090
2,insurance,0.057
3,link,0.057
4,calling,0.053
5,call,0.050
6,paid,0.049
7,service,0.048
8,time,0.047
9,entry,0.044



🟢 Legitimate-pushing signals:


,feature,contribution
0,marketing,-0.129
1,new,-0.085
2,medical,-0.054
3,affordable,-0.051
4,sales representative,-0.046
5,days,-0.041
6,care,-0.040
7,our employees,-0.037
8,about,-0.032
9,employment,-0.031


In [78]:
# ============================================
# TEST EXPLAINABILITY ON AN ACTUAL FRAUD JOB
# ============================================

# Find the first actual fraudulent job in the test set
fraud_indices = y_test[y_test == 1].index

fraud_sample_index = fraud_indices[0]

# Generate explanation
fraud_explanation = get_job_explanation(
    fraud_sample_index,
    top_n=10
)

print("============================================")
print("ACTUAL FRAUD JOB — JOBSHIELD EXPLANATION")
print("============================================")

print(
    "\nActual label: FRAUD"
)

print(
    "Model prediction:",
    fraud_explanation["prediction"]
)

print(
    "Fraud probability:",
    round(
        fraud_explanation["probability"],
        3
    )
)

print(
    "Decision threshold:",
    final_threshold
)

print("\n🔴 Fraud-pushing signals:")
display(
    fraud_explanation["fraud_signals"].round(3)
)

print("\n🟢 Legitimate-pushing signals:")
display(
    fraud_explanation["legit_signals"].round(3)
)

ACTUAL FRAUD JOB — JOBSHIELD EXPLANATION

Actual label: FRAUD
Model prediction: Potentially Fraudulent
Fraud probability: 0.837
Decision threshold: 0.6

🔴 Fraud-pushing signals:


,feature,contribution
0,vam systems,0.243
1,vam,0.243
2,qatar,0.228
3,systems,0.160
4,and exchange,0.126
5,ms,0.097
6,change management,0.078
7,exchange,0.074
8,perform,0.067
9,administrator for,0.061



🟢 Legitimate-pushing signals:


,feature,contribution
0,services,-0.041
1,administration,-0.040
2,infrastructure,-0.038
3,solutions,-0.034
4,management,-0.027
5,directory,-0.026
6,experience,-0.025
7,tasks,-0.022
8,processes,-0.021
9,active directory,-0.020


In [79]:
# ============================================
# JOBSHIELD — EXPLANATION QUALITY CHECK
# ============================================

def summarize_job(index):
    """Return prediction + strongest model signals for one job."""

    test_position = list(X_test.index).index(index)

    sample_vector = X_test_final_tfidf[test_position]

    probability = final_model.predict_proba(
        sample_vector
    )[0, 1]

    prediction = (
        "Fraud"
        if probability >= final_threshold
        else "Legitimate"
    )

    feature_values = sample_vector.toarray().flatten()

    contributions = feature_values * coefficients

    explanation = pd.DataFrame({
        "feature": feature_names,
        "contribution": contributions
    })

    explanation = explanation[
        explanation["contribution"] != 0
    ]

    # Remove common/unhelpful words
    explanation["words"] = (
        explanation["feature"]
        .str.lower()
        .str.split()
    )

    explanation = explanation[
        ~explanation["words"].apply(
            lambda words: all(
                word in stop_words_for_explanation
                for word in words
            )
        )
    ]

    fraud_signals = (
        explanation[
            explanation["contribution"] > 0
        ]
        .sort_values("contribution", ascending=False)
        .head(5)["feature"]
        .tolist()
    )

    legit_signals = (
        explanation[
            explanation["contribution"] < 0
        ]
        .sort_values("contribution")
        .head(5)["feature"]
        .tolist()
    )

    return {
        "actual": "Fraud" if y_test.loc[index] == 1 else "Legitimate",
        "prediction": prediction,
        "probability": round(probability, 3),
        "fraud_signals": ", ".join(fraud_signals),
        "legit_signals": ", ".join(legit_signals)
    }


# --------------------------------------------
# Select 5 fraud + 5 legitimate jobs
# --------------------------------------------

fraud_test_indices = y_test[y_test == 1].index[:5]
legit_test_indices = y_test[y_test == 0].index[:5]

selected_indices = list(fraud_test_indices) + list(
    legit_test_indices
)


# --------------------------------------------
# Generate explanations
# --------------------------------------------

explanation_results = []

for index in selected_indices:

    result = summarize_job(index)
    result["index"] = index

    explanation_results.append(result)


explanation_results = pd.DataFrame(
    explanation_results
)

explanation_results = explanation_results[
    [
        "index",
        "actual",
        "prediction",
        "probability",
        "fraud_signals",
        "legit_signals"
    ]
]

print("============================================")
print("JOBSHIELD EXPLANATION QUALITY CHECK")
print("============================================")

display(explanation_results)

JOBSHIELD EXPLANATION QUALITY CHECK


,index,actual,prediction,probability,fraud_signals,legit_signals
0,3274,Fraud,Fraud,0.837,"vam systems, vam, qatar, systems, and exchange","services, administration, infrastructure, solu..."
1,2927,Fraud,Legitimate,0.314,"lean, skills, manager, engineering, change man...","green, etc, experience, enterprise, end"
2,17666,Fraud,Fraud,0.963,"service representative, customer service, insu...","medical, experience with, health, to provide, ..."
3,17780,Fraud,Legitimate,0.221,"manager, project manager, project","de, junior"
4,17720,Fraud,Fraud,0.759,"assistant, requirements, ms word, part time, ms","finance, growing, client, fast growing, software"
5,9571,Legitimate,Legitimate,0.095,"company, discounts, insurance, link, calling","marketing, new, medical, affordable, sales rep..."
6,11429,Legitimate,Legitimate,0.101,"clerk, per, service, six, time","pcp, pride, about, legal, team"
7,16553,Legitimate,Legitimate,0.426,"information, information security, systems, re...","security, experience, software, about, knowled..."
8,13741,Legitimate,Legitimate,0.013,"apply, get, contract, url, get paid","english, university, abroad, url url, love"
9,9083,Legitimate,Legitimate,0.023,"customer, achieve, employee, reps, manager","companies, clients, marketing, leading, european"


In [80]:
# ============================================
# JOBSHIELD — FALSE NEGATIVE ANALYSIS
# ============================================

# Find fraudulent jobs that the final model classified as legitimate
false_negative_indices = y_test[
    (y_test == 1) & (y_test_pred == 0)
].index

print("Actual fraudulent jobs:", int(y_test.sum()))
print("Fraudulent jobs detected:", int(((y_test == 1) & (y_test_pred == 1)).sum()))
print("Fraudulent jobs missed:", len(false_negative_indices))


# --------------------------------------------
# Build a table of the missed fraud cases
# --------------------------------------------

false_negative_results = []

for index in false_negative_indices:

    test_position = list(X_test.index).index(index)

    probability = y_test_prob[test_position]

    result = summarize_job(index)

    false_negative_results.append({
        "index": index,
        "fraud_probability": round(probability, 3),
        "prediction": result["prediction"],
        "fraud_signals": result["fraud_signals"],
        "legit_signals": result["legit_signals"]
    })


false_negative_results = pd.DataFrame(
    false_negative_results
).sort_values(
    "fraud_probability",
    ascending=False
)


# --------------------------------------------
# Display the closest-to-being-detected cases
# --------------------------------------------

print("\n============================================")
print("FALSE NEGATIVES — CLOSEST TO THRESHOLD")
print("============================================")

display(
    false_negative_results.head(15)
)


# --------------------------------------------
# Probability distribution of missed fraud
# --------------------------------------------

print("\n============================================")
print("FALSE NEGATIVE PROBABILITY SUMMARY")
print("============================================")

print(
    false_negative_results["fraud_probability"].describe().round(3)
)

Actual fraudulent jobs: 172
Fraudulent jobs detected: 138
Fraudulent jobs missed: 34

FALSE NEGATIVES — CLOSEST TO THRESHOLD


,index,fraud_probability,prediction,fraud_signals,legit_signals
9,17785,0.581,Legitimate,"earn, from home, trading, work from, part time","financial, year, capital, marketing, dedicated"
23,2969,0.568,Legitimate,"supplier, line requirements, engineering, bott...","materials, sourcing, experience, strong, distr..."
11,6974,0.531,Legitimate,"engineering, drilling, project, skills, oil","experience, delivery, management, team of, team"
13,16860,0.524,Legitimate,"engineering, equipment, project, six sigma, sigma","new, experience, capital, marketing, process"
32,17724,0.521,Legitimate,"clerk, per, accounting, hour, per hour","accounts, finance, payable, accounts payable, ..."
28,17801,0.514,Legitimate,"assistant, administrative, administrative assi...","executive assistant, candidate should, experie..."
22,17812,0.510,Legitimate,"supply, systems, sap, to align, manager","chain, team, solutions, supply chain, relation..."
31,17626,0.490,Legitimate,"migration, center, data, be experienced, lead","data center, application, experience, infrastr..."
19,6984,0.479,Legitimate,"expro, subsea, appraisal, offshore, standards","process, resources, human, hr, services"
4,16552,0.479,Legitimate,"per diem, diem, per, rn, phone","care, home care, therapy, providing, one"



FALSE NEGATIVE PROBABILITY SUMMARY
count    34.000
mean      0.367
std       0.134
min       0.129
25%       0.278
50%       0.352
75%       0.479
max       0.581
Name: fraud_probability, dtype: float64


In [81]:
# ============================================
# JOBSHIELD — FALSE NEGATIVE CATEGORY ANALYSIS
# ============================================

# Get the original structured information for the missed fraud jobs
false_negative_data = df_clean.loc[
    false_negative_indices,
    [
        "title",
        "location",
        "employment_type",
        "required_experience",
        "required_education",
        "industry",
        "function",
        "telecommuting",
        "has_company_logo",
        "has_questions"
    ]
].copy()


# Add model probability
false_negative_data["fraud_probability"] = [
    y_test_prob[list(X_test.index).index(i)]
    for i in false_negative_indices
]


# ============================================
# 1. Industry distribution
# ============================================

print("============================================")
print("MISSED FRAUD — INDUSTRY")
print("============================================")

industry_counts = (
    false_negative_data["industry"]
    .fillna("MISSING")
    .value_counts()
)

display(industry_counts.head(15).to_frame("count"))


# ============================================
# 2. Job function distribution
# ============================================

print("\n============================================")
print("MISSED FRAUD — FUNCTION")
print("============================================")

function_counts = (
    false_negative_data["function"]
    .fillna("MISSING")
    .value_counts()
)

display(function_counts.head(15).to_frame("count"))


# ============================================
# 3. Employment type
# ============================================

print("\n============================================")
print("MISSED FRAUD — EMPLOYMENT TYPE")
print("============================================")

employment_counts = (
    false_negative_data["employment_type"]
    .fillna("MISSING")
    .value_counts()
)

display(employment_counts.to_frame("count"))


# ============================================
# 4. Required experience
# ============================================

print("\n============================================")
print("MISSED FRAUD — EXPERIENCE")
print("============================================")

experience_counts = (
    false_negative_data["required_experience"]
    .fillna("MISSING")
    .value_counts()
)

display(experience_counts.to_frame("count"))


# ============================================
# 5. Remote / telecommuting
# ============================================

print("\n============================================")
print("MISSED FRAUD — TELECOMMUTING")
print("============================================")

telecommuting_counts = (
    false_negative_data["telecommuting"]
    .map({0: "No", 1: "Yes"})
    .fillna("MISSING")
    .value_counts()
)

display(telecommuting_counts.to_frame("count"))


# ============================================
# 6. Company logo
# ============================================

print("\n============================================")
print("MISSED FRAUD — COMPANY LOGO")
print("============================================")

logo_counts = (
    false_negative_data["has_company_logo"]
    .map({0: "No", 1: "Yes"})
    .fillna("MISSING")
    .value_counts()
)

display(logo_counts.to_frame("count"))


# ============================================
# 7. Screening questions
# ============================================

print("\n============================================")
print("MISSED FRAUD — SCREENING QUESTIONS")
print("============================================")

question_counts = (
    false_negative_data["has_questions"]
    .map({0: "No", 1: "Yes"})
    .fillna("MISSING")
    .value_counts()
)

display(question_counts.to_frame("count"))


# ============================================
# 8. Titles of missed fraud jobs
# ============================================

print("\n============================================")
print("MISSED FRAUD — JOB TITLES")
print("============================================")

display(
    false_negative_data[
        ["title", "industry", "function", "fraud_probability"]
    ]
    .sort_values("fraud_probability", ascending=False)
    .head(20)
)

MISSED FRAUD — INDUSTRY


,count
industry,
MISSING,10
Oil & Energy,4
Information Technology and Services,4
Marketing and Advertising,3
Accounting,2
Consumer Services,2
Hospital & Health Care,2
Airlines/Aviation,1
Design,1



MISSED FRAUD — FUNCTION


,count
function,
MISSING,12
Information Technology,5
Project Management,2
Other,2
Engineering,2
Customer Service,2
Art/Creative,1
Finance,1
Education,1



MISSED FRAUD — EMPLOYMENT TYPE


...

,count
has_company_logo,
No,29
Yes,5



MISSED FRAUD — SCREENING QUESTIONS


,count
has_questions,
No,21
Yes,13



MISSED FRAUD — JOB TITLES


,title,industry,function,fraud_probability
17785,work from home hr,Accounting,Finance,0.581241
2969,strategic sourcing engineer 2053,NaN,NaN,0.567719
6974,manager of project management organization eng...,Oil & Energy,Project Management,0.531018
16860,process validation project manager,Biotechnology,Engineering,0.523597
17724,accounts payable clerk,Accounting,Accounting/Auditing,0.521372
17801,administrative position,Consumer Services,Administrative,0.513879
17812,hiring for sap supply chain manager,Information Technology and Services,Information Technology,0.509668
17626,data center migration app lead for full time o...,Information Technology and Services,Information Technology,0.490341
6984,hr process leader,Oil & Energy,Human Resources,0.479225
16552,nursing therapy positions home care,NaN,NaN,0.479001


In [82]:
# ============================================
# JOBSHIELD — TARGETED HYBRID MODEL
# ============================================

from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack
from sklearn.metrics import classification_report, average_precision_score

# --------------------------------------------
# 1. Define targeted structured features
# --------------------------------------------

structured_columns = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
    "company_profile_missing",
    "requirements_missing",
    "benefits_missing",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

structured_data = df_clean[structured_columns].copy()

# Fill categorical missing values
categorical_columns_targeted = [
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

structured_data[categorical_columns_targeted] = (
    structured_data[categorical_columns_targeted]
    .fillna("MISSING")
)

# Make sure binary fields are numeric
binary_columns_targeted = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
    "company_profile_missing",
    "requirements_missing",
    "benefits_missing"
]

for column in binary_columns_targeted:
    structured_data[column] = (
        structured_data[column]
        .fillna(0)
        .astype(int)
    )


# --------------------------------------------
# 2. Create train / validation / test splits
# --------------------------------------------

structured_train = structured_data.loc[X_train_final.index]
structured_val = structured_data.loc[X_val.index]

# --------------------------------------------
# 3. Encode categorical variables
# --------------------------------------------

encoder_targeted = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_struct_train = encoder_targeted.fit_transform(
    structured_train
)

X_struct_val = encoder_targeted.transform(
    structured_val
)

print("Structured training shape:", X_struct_train.shape)
print("Structured validation shape:", X_struct_val.shape)


# --------------------------------------------
# 4. Combine with TF-IDF
# --------------------------------------------

X_train_targeted_hybrid = hstack([
    X_train_final_tfidf,
    X_struct_train
])

X_val_targeted_hybrid = hstack([
    X_val_tfidf,
    X_struct_val
])

print(
    "Combined training shape:",
    X_train_targeted_hybrid.shape
)

print(
    "Combined validation shape:",
    X_val_targeted_hybrid.shape
)


# --------------------------------------------
# 5. Train targeted hybrid model
# --------------------------------------------

targeted_hybrid_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

targeted_hybrid_model.fit(
    X_train_targeted_hybrid,
    y_train_final
)


# --------------------------------------------
# 6. Validation probabilities
# --------------------------------------------

y_targeted_val_prob = (
    targeted_hybrid_model
    .predict_proba(X_val_targeted_hybrid)[:, 1]
)


# --------------------------------------------
# 7. PR-AUC
# --------------------------------------------

targeted_pr_auc = average_precision_score(
    y_val,
    y_targeted_val_prob
)

print("\nTargeted Hybrid Validation PR-AUC:",
      round(targeted_pr_auc, 3))


# --------------------------------------------
# 8. Find best threshold on validation
# --------------------------------------------

thresholds = np.arange(0.10, 0.91, 0.05)

targeted_results = []

for threshold in thresholds:

    y_pred = (
        y_targeted_val_prob >= threshold
    ).astype(int)

    report = classification_report(
        y_val,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    targeted_results.append({
        "threshold": round(threshold, 2),
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1": report["1"]["f1-score"]
    })

targeted_results = pd.DataFrame(
    targeted_results
)

print("\nTargeted Hybrid Threshold Results:")
display(
    targeted_results.round(3)
)


# --------------------------------------------
# 9. Best validation threshold
# --------------------------------------------

best_targeted = targeted_results.loc[
    targeted_results["f1"].idxmax()
]

print("\nBest Targeted Hybrid Threshold:",
      best_targeted["threshold"])

print("Precision:",
      round(best_targeted["precision"], 3))

print("Recall:",
      round(best_targeted["recall"], 3))

print("F1:",
      round(best_targeted["f1"], 3))


# --------------------------------------------
# 10. Compare against our current champion
# --------------------------------------------

print("\n============================================")
print("MODEL COMPARISON")
print("============================================")

comparison = pd.DataFrame({
    "model": [
        "TF-IDF + Logistic Regression",
        "Targeted Hybrid"
    ],
    "validation_PR_AUC": [
        0.933,
        targeted_pr_auc
    ],
    "validation_best_F1": [
        0.881,
        best_targeted["f1"]
    ],
    "validation_best_threshold": [
        0.60,
        best_targeted["threshold"]
    ]
})

display(comparison.round(3))

Structured training shape: (11292, 206)
Structured validation shape: (2824, 206)
Combined training shape: (11292, 50206)
Combined validation shape: (2824, 50206)

Targeted Hybrid Validation PR-AUC: 0.862

Targeted Hybrid Threshold Results:


,threshold,precision,recall,f1
0,0.10,0.217,0.985,0.356
1,0.15,0.272,0.978,0.426
2,0.20,0.305,0.956,0.462
3,0.25,0.351,0.949,0.513
4,0.30,0.411,0.942,0.572
5,0.35,0.456,0.934,0.612
6,0.40,0.500,0.934,0.651
7,0.45,0.559,0.927,0.698
8,0.50,0.606,0.898,0.724
9,0.55,0.665,0.883,0.759



Best Targeted Hybrid Threshold: 0.85
Precision: 0.907
Recall: 0.708
F1: 0.795

MODEL COMPARISON


,model,validation_PR_AUC,validation_best_F1,validation_best_threshold
0,TF-IDF + Logistic Regression,0.933,0.881,0.60
1,Targeted Hybrid,0.862,0.795,0.85


In [84]:
# ============================================
# JOBSHIELD — IMPROVED EVIDENCE LAYER
# ============================================

def get_structured_evidence(sample_index, top_n=5):

    # Locate job in test set
    test_position = list(X_test.index).index(sample_index)

    sample_vector = X_test_final_tfidf[test_position]

    # ----------------------------------------
    # Prediction
    # ----------------------------------------

    probability = final_model.predict_proba(
        sample_vector
    )[0, 1]

    prediction = (
        "Potentially Fraudulent"
        if probability >= final_threshold
        else "Likely Legitimate"
    )

    # ----------------------------------------
    # Feature contributions
    # ----------------------------------------

    feature_values = sample_vector.toarray().flatten()

    contributions = feature_values * coefficients

    explanation = pd.DataFrame({
        "feature": feature_names,
        "contribution": contributions
    })

    # Only features actually present
    explanation = explanation[
        explanation["contribution"] != 0
    ].copy()

    # Remove common words
    explanation["words"] = (
        explanation["feature"]
        .str.lower()
        .str.split()
    )

    explanation = explanation[
        ~explanation["words"].apply(
            lambda words: all(
                word in stop_words_for_explanation
                for word in words
            )
        )
    ].copy()

    # ----------------------------------------
    # Categorize only meaningful signals
    # ----------------------------------------

    evidence = {}

    for category in evidence_categories:
        evidence[category] = {
            "fraud": [],
            "legitimate": []
        }

    for _, row in explanation.iterrows():

        feature = row["feature"]
        contribution = float(row["contribution"])

        categories = classify_signal(feature)

        for category in categories:

            signal = {
                "feature": feature,
                "contribution": round(
                    contribution, 3
                )
            }

            if contribution > 0:
                evidence[category]["fraud"].append(
                    signal
                )

            elif contribution < 0:
                evidence[category]["legitimate"].append(
                    signal
                )

    # ----------------------------------------
    # Sort and keep strongest signals
    # ----------------------------------------

    for category in evidence:

        evidence[category]["fraud"] = sorted(
            evidence[category]["fraud"],
            key=lambda x: x["contribution"],
            reverse=True
        )[:top_n]

        evidence[category]["legitimate"] = sorted(
            evidence[category]["legitimate"],
            key=lambda x: x["contribution"]
        )[:top_n]

    return {
        "prediction": prediction,
        "probability": round(
            float(probability), 3
        ),
        "threshold": final_threshold,
        "evidence": evidence
    }


# ============================================
# TEST
# ============================================

structured_evidence = get_structured_evidence(
    fraud_sample_index
)

print("============================================")
print("IMPROVED JOBSHIELD EVIDENCE")
print("============================================")

print(
    "\nPrediction:",
    structured_evidence["prediction"]
)

print(
    "Fraud probability:",
    structured_evidence["probability"]
)

print(
    "Threshold:",
    structured_evidence["threshold"]
)

for category, signals in structured_evidence[
    "evidence"
].items():

    if signals["fraud"]:

        print(f"\n🔴 {category} — fraud signals")

        for signal in signals["fraud"]:
            print(
                f"   {signal['feature']} "
                f"({signal['contribution']:+.3f})"
            )

    if signals["legitimate"]:

        print(
            f"\n🟢 {category} — "
            "legitimate signals"
        )

        for signal in signals["legitimate"]:
            print(
                f"   {signal['feature']} "
                f"({signal['contribution']:+.3f})"
            )

IMPROVED JOBSHIELD EVIDENCE

Prediction: Potentially Fraudulent
Fraud probability: 0.837
Threshold: 0.6

🔴 external_application — fraud signals
   apply (+0.018)

🟢 external_application — legitimate signals
   application (-0.014)
   applications (-0.009)
   and apply (-0.002)
   applications including (-0.002)

🔴 phone_contact — fraud signals
   phone (+0.044)
   call us (+0.038)
   call (+0.032)
   or call (+0.017)

🔴 data_entry — fraud signals
   database (+0.005)

🔴 professional_business — fraud signals
   change management (+0.078)
   company (+0.013)
   retail management (+0.013)
   is business (+0.008)
   management resources (+0.008)

🟢 professional_business — legitimate signals
   management (-0.027)
   team (-0.014)
   management and (-0.010)
   company with (-0.005)
   services company (-0.005)


In [85]:
# ============================================
# JOBSHIELD — CLEAN MODEL EVIDENCE
# ============================================

def get_clean_model_evidence(sample_index, top_n=10):

    # Find job position inside X_test
    test_position = list(X_test.index).index(sample_index)

    sample_vector = X_test_final_tfidf[test_position]

    # ----------------------------------------
    # Prediction
    # ----------------------------------------

    probability = final_model.predict_proba(
        sample_vector
    )[0, 1]

    prediction = (
        "Potentially Fraudulent"
        if probability >= final_threshold
        else "Likely Legitimate"
    )

    # ----------------------------------------
    # Calculate feature contributions
    # ----------------------------------------

    feature_values = sample_vector.toarray().flatten()

    contributions = feature_values * coefficients

    evidence = pd.DataFrame({
        "feature": feature_names,
        "contribution": contributions
    })

    # Keep only features actually present
    evidence = evidence[
        evidence["contribution"] != 0
    ].copy()

    # ----------------------------------------
    # Remove common/unhelpful words
    # ----------------------------------------

    evidence["words"] = (
        evidence["feature"]
        .str.lower()
        .str.split()
    )

    evidence = evidence[
        ~evidence["words"].apply(
            lambda words: all(
                word in stop_words_for_explanation
                for word in words
            )
        )
    ].copy()

    # ----------------------------------------
    # Strongest fraud evidence
    # ----------------------------------------

    fraud_evidence = (
        evidence[
            evidence["contribution"] > 0
        ]
        .sort_values(
            "contribution",
            ascending=False
        )
        .head(top_n)
        [["feature", "contribution"]]
        .copy()
    )

    # ----------------------------------------
    # Strongest legitimate evidence
    # ----------------------------------------

    legitimate_evidence = (
        evidence[
            evidence["contribution"] < 0
        ]
        .sort_values(
            "contribution",
            ascending=True
        )
        .head(top_n)
        [["feature", "contribution"]]
        .copy()
    )

    # ----------------------------------------
    # Convert to simple records
    # ----------------------------------------

    fraud_records = [
        {
            "feature": row["feature"],
            "contribution": round(
                float(row["contribution"]), 3
            )
        }
        for _, row in fraud_evidence.iterrows()
    ]

    legitimate_records = [
        {
            "feature": row["feature"],
            "contribution": round(
                float(row["contribution"]), 3
            )
        }
        for _, row in legitimate_evidence.iterrows()
    ]

    # ----------------------------------------
    # Final evidence object
    # ----------------------------------------

    return {
        "prediction": prediction,
        "fraud_probability": round(
            float(probability), 3
        ),
        "threshold": final_threshold,
        "fraud_evidence": fraud_records,
        "legitimate_evidence": legitimate_records
    }


# ============================================
# TEST
# ============================================

clean_evidence = get_clean_model_evidence(
    fraud_sample_index
)

print("============================================")
print("JOBSHIELD CLEAN MODEL EVIDENCE")
print("============================================")

print(
    "\nPrediction:",
    clean_evidence["prediction"]
)

print(
    "Fraud probability:",
    clean_evidence["fraud_probability"]
)

print(
    "Decision threshold:",
    clean_evidence["threshold"]
)

print("\n🔴 Fraud evidence:")
for item in clean_evidence["fraud_evidence"]:
    print(
        f"   {item['feature']} "
        f"({item['contribution']:+.3f})"
    )

print("\n🟢 Legitimate evidence:")
for item in clean_evidence["legitimate_evidence"]:
    print(
        f"   {item['feature']} "
        f"({item['contribution']:+.3f})"
    )

JOBSHIELD CLEAN MODEL EVIDENCE

Prediction: Potentially Fraudulent
Fraud probability: 0.837
Decision threshold: 0.6

🔴 Fraud evidence:
   vam systems (+0.243)
   vam (+0.243)
   qatar (+0.228)
   systems (+0.160)
   and exchange (+0.126)
   ms (+0.097)
   change management (+0.078)
   exchange (+0.074)
   perform (+0.067)
   administrator for (+0.061)

🟢 Legitimate evidence:
   services (-0.041)
   administration (-0.040)
   infrastructure (-0.038)
   solutions (-0.034)
   management (-0.027)
   directory (-0.026)
   experience (-0.025)
   tasks (-0.022)
   processes (-0.021)
   active directory (-0.020)


In [86]:
# ============================================
# JOBSHIELD — SAVE FINAL MODEL ARTIFACTS
# ============================================

import joblib
from pathlib import Path

# Make sure the models directory exists
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

# Save TF-IDF vectorizer
joblib.dump(
    final_tfidf,
    models_dir / "jobshield_tfidf.pkl"
)

# Save trained Logistic Regression model
joblib.dump(
    final_model,
    models_dir / "jobshield_model.pkl"
)

print("✅ Model artifacts saved successfully!")

print("\nSaved files:")
print(models_dir / "jobshield_tfidf.pkl")
print(models_dir / "jobshield_model.pkl")

✅ Model artifacts saved successfully!

Saved files:
..\models\jobshield_tfidf.pkl
..\models\jobshield_model.pkl
